# 外部操作已经成功，但程序崩溃了，还能直接重试吗？

## V0.3 Durable Tool WAL / Reconciliation

这是本组 lab 的 flagship。危险点不是“程序崩溃”，而是外部系统已经成功扣款，本地还没写入 COMMIT。

**Core:** Recovery != Retry.

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "agentkernel").is_dir():
            return path
    raise RuntimeError("Run this notebook from inside the AgentKernel repository.")

REPOSITORY_ROOT = find_repo_root()
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))
LABS_ROOT = REPOSITORY_ROOT / "examples" / "labs"
if str(LABS_ROOT) not in sys.path:
    sys.path.insert(0, str(LABS_ROOT))

from lab_helpers import event_rows, grant_rows, print_table, process_row, trajectory

## 1. Fake external payment service

In [ ]:
import asyncio
import tempfile
from collections.abc import Mapping
from pathlib import Path

from agentkernel import (
    Agent, DurableToolExecutor, EventType, JsonlSessionPersistence,
    OperationRecoveryClassification, ReconcileResult, ReconcileStatus,
    Session, ToolCall, ToolDefinition, ToolEffectKind, ToolExecutionContext,
    ToolRegistry, ToolSchema,
)
from agentkernel.protocol import JsonValue

class FakePaymentService:
    def __init__(self) -> None:
        self.effects: dict[str, JsonValue] = {}
        self.external_effect_count = 0

    async def charge(self, arguments: Mapping[str, JsonValue], context: ToolExecutionContext) -> JsonValue:
        existing = self.effects.get(context.operation_id)
        if existing is not None:
            return existing
        self.external_effect_count += 1
        output: JsonValue = {
            "payment_id": f"pay-{self.external_effect_count}",
            "amount": arguments["amount"],
            "currency": arguments["currency"],
        }
        self.effects[context.operation_id] = output
        return output

    async def reconcile(self, context: ToolExecutionContext) -> ReconcileResult:
        output = self.effects.get(context.operation_id)
        if output is None:
            return ReconcileResult(ReconcileStatus.NOT_FOUND)
        return ReconcileResult(ReconcileStatus.SUCCEEDED, output=output)

def payment_registry(service: FakePaymentService) -> ToolRegistry:
    tools = ToolRegistry()
    tools.register(ToolDefinition(
        schema=ToolSchema("payment.charge", "Charge a fake payment.", {"type": "object"}),
        handler=service.charge,
        required_capability="payment.charge",
        effect_kind=ToolEffectKind.RECONCILABLE_MUTATION,
        reconcile_handler=service.reconcile,
    ))
    return tools

def auth_context() -> dict[str, JsonValue]:
    return {
        "agent_id": "lab-v0-3-agent",
        "action": "tool.execute",
        "resource_scope": "tool://payment.charge",
        "reason": "allowed",
        "matched_grant": {
            "subject": "lab-v0-3-agent",
            "action": "tool.execute",
            "resource_scope": "tool://payment.charge",
        },
    }

## 2. PREPARE -> DISPATCH is written before the external effect

In [ ]:
tmpdir = tempfile.TemporaryDirectory(prefix="agentkernel-lab-v0-3-")
path = Path(tmpdir.name) / "payment-session.jsonl"
service = FakePaymentService()
call = ToolCall("model-call-payment-1", "payment.charge", {"amount": 42, "currency": "USD"})
session = Session("lab-v0-3-session", JsonlSessionPersistence(path))
auth = auth_context()
session.append(EventType.TURN_START, {"turn": 1})
session.append(EventType.USER_MESSAGE, {"turn": 1, "content": "Charge the fake invoice."})
session.append(EventType.STEP_START, {"turn": 1, "step": 1})
session.append(EventType.ASSISTANT_MESSAGE, {"turn": 1, "step": 1, "content": "", "tool_calls": [call.as_dict()]})
session.append(EventType.TOOL_CALL, {"turn": 1, "step": 1, **call.as_dict()})
session.append(EventType.AUTHORIZATION_GRANTED, {**auth, "turn": 1, "step": 1, "tool_call_id": call.call_id, "tool_name": call.name, "operation_id": "payment-op-1", "boundary": "prepare"})
session.append(EventType.TOOL_PREPARE, {"turn": 1, "step": 1, "operation_id": "payment-op-1", "tool_call_id": call.call_id, "tool_name": call.name, "effect_kind": ToolEffectKind.RECONCILABLE_MUTATION.value, "authorization": auth})
session.flush()
session.append(EventType.AUTHORIZATION_GRANTED, {**auth, "turn": 1, "step": 1, "tool_call_id": call.call_id, "tool_name": call.name, "operation_id": "payment-op-1", "boundary": "dispatch"})
session.append(EventType.TOOL_DISPATCH, {"turn": 1, "step": 1, "operation_id": "payment-op-1", "attempt": 1, "authorization": auth})
session.flush()
print_table(event_rows(session)[-4:])

## 3. External success happens, then the runtime crashes before COMMIT

In [ ]:
result = asyncio.run(service.charge(
    call.arguments,
    ToolExecutionContext(
        agent_id="lab-v0-3-agent",
        session_id=session.session_id,
        tool_call_id=call.call_id,
        operation_id="payment-op-1",
    ),
))
session.close()
print_table([
    {"stage": "external service", "value": result},
    {"stage": "external effect count", "value": service.external_effect_count},
    {"stage": "local commit written?", "value": False},
])

## 4. Restart: WAL says reconcile, not blind retry

In [ ]:
restored = Session.load("lab-v0-3-session", JsonlSessionPersistence(path))
operation = restored.recovery_analysis.durable_operations[0]
print_table([
    {"fact": "operation_id", "value": operation.operation_id},
    {"fact": "classification", "value": operation.classification.value},
    {"fact": "committed", "value": operation.committed},
])
assert operation.classification is OperationRecoveryClassification.RECONCILE_REQUIRED

## 5. Reconcile the already-successful external effect and commit it once

In [ ]:
agent = Agent.create(agent_id="lab-v0-3-agent", session=restored, capabilities={"payment.charge"})
observed = asyncio.run(DurableToolExecutor(payment_registry(service)).reconcile(operation, agent.control, restored))
final = restored.recovery_analysis.durable_operations[0]
print_table([
    {"fact": "reconcile status", "value": observed.status.value},
    {"fact": "final classification", "value": final.classification.value},
    {"fact": "external effect count", "value": service.external_effect_count},
    {"fact": "committed", "value": final.committed},
])
trajectory("PREPARE", "DISPATCH", "external success", "crash before COMMIT", "restart", "RECONCILE_REQUIRED", "commit existing result")
restored.close()
tmpdir.cleanup()

## Invariant

After dispatch, recovery must first reconcile durable operation state; blind retry can duplicate the external side effect.

## WHAT THIS DEMONSTRATES / 本实验验证什么

- WAL records `PREPARE` and `DISPATCH` before local completion.
- Crash after dispatch is classified as `reconcile_required`.
- Reconciliation commits the observed fake external effect once.

## WHAT THIS DOES NOT DEMONSTRATE / 本实验不证明什么

- It does not use a real payment provider.
- It does not prove universal exactly-once semantics.
- It does not prove arbitrary external-system atomicity.